This is a brief demonstration of the workflow for the quantum protein folding problem on the FCC lattice. In the following sections, we present two key methods for building and solving the FCC Hamiltonian: polynomial fitting and the Variational Quantum Eigensolver with Constraints (VQEC) based on the Lagrangian dual method.

## Configuration

Edit this cell to configure the complete workflow. Hardware execution is disabled by default.

In [ ]:
# Primary execution mode. Use "simulator" to develop and create a warm
# start, then change this to "hardware" for QPU refinement.
EXECUTION_MODE = "simulator"  # "simulator" or "hardware"

# Warm starts are compressed NumPy files containing optimized PolyFit and
# VQEC parameters plus compatibility metadata. Hardware mode loads this
# file by default; incompatible sequences or ansatz settings are rejected.
WARM_START_FILE = "fcc_warm_start.npz"
LOAD_WARM_START = EXECUTION_MODE == "hardware"
SAVE_WARM_START = EXECUTION_MODE == "simulator"

# Parallel processing affects classical post-processing only. Python
# multiprocessing is portable; Ray is an optional accelerator.
PARALLELIZER = "auto"  # "auto", "serial", "python-mp", or "ray"
NUM_WORKERS = None  # None uses the number of physical CPU cores

# PolyFit workflow
POLYFIT_PROTEIN_SEQUENCE = "GNLVS"
POLYFIT_PENALTY_BACK = 100.0
POLYFIT_PENALTY_REDUNDANCY = 100.0
POLYFIT_PENALTY_OVERLAP = 100.0
POLYFIT_R2_THRESHOLD = 1.0
POLYFIT_CHUNK_SIZE = 20
POLYFIT_ANSATZ_REPS = 1
POLYFIT_SHOTS = 10_000
POLYFIT_OPTIMIZER = "COBYLA"
POLYFIT_SIMULATOR_MAX_ITER = 200
# Each hardware objective evaluation submits a Runtime sampler job. COBYLA
# may perform more function evaluations than this nominal iteration value.
POLYFIT_HARDWARE_MAX_ITER = 10

# Chance-constrained VQEC workflow
VQEC_PROTEIN_SEQUENCE = "GNLVS"
VQEC_PENALTY_BACK = 100.0
VQEC_PENALTY_REDUNDANCY = 100.0
VQEC_CONSTRAINT_LIMIT = 0.01
VQEC_ANSATZ_REPS = 2
VQEC_SHOTS = 10_000
VQEC_RANDOM_SEED = 7
VQEC_PERTURB_STEP = 0.05
VQEC_GAMMA = 0.1
VQEC_SIMULATOR_MAX_ITER = 100
# Every hardware VQEC iteration samples the current/perturbed states and a
# batch of two parameter-shift circuits per ansatz parameter. Start small.
VQEC_HARDWARE_MAX_ITER = 5

# IBM Quantum hardware settings. None selects the least-busy operational
# QPU with enough qubits. Shots apply to every hardware sampler call.
HARDWARE_BACKEND = None  # None selects the least-busy suitable QPU
HARDWARE_MIN_QUBITS = 20  # Must cover the largest configured workflow
HARDWARE_SHOTS = 1_000
HARDWARE_OPTIMIZATION_LEVEL = 1

## Execution backend and warm-start setup

This cell validates the selected mode, prepares the simulator or IBM Runtime backend, and loads a compatible warm start when requested. Hardware mode drives the full workflows below; it is not a separate smoke test.

In [ ]:
import json
import numpy as np
from pathlib import Path
from fcc import MiyazawaJerniganInteraction, Peptide, PenaltyParameters, ProteinFoldingProblem, ProteinSolver
from qiskit.circuit.library import real_amplitudes
from qiskit_aer.primitives import SamplerV2 as AerSampler

if EXECUTION_MODE not in {"simulator", "hardware"}:
    raise ValueError("EXECUTION_MODE must be 'simulator' or 'hardware'")

warm_start = {}
warm_start_metadata = {}
warm_start_path = Path(WARM_START_FILE)
if LOAD_WARM_START:
    if not warm_start_path.is_file():
        raise FileNotFoundError(
            f"Warm-start file not found: {warm_start_path}. Run simulator mode first."
        )
    with np.load(warm_start_path, allow_pickle=False) as saved:
        warm_start_metadata = json.loads(str(saved["metadata"]))
        warm_start = {
            key: np.array(saved[key], dtype=float)
            for key in saved.files
            if key != "metadata"
        }
    print(f"Loaded warm start from {warm_start_path}")

backend = None
pass_manager = None
if EXECUTION_MODE == "hardware":
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler

    service = QiskitRuntimeService()
    backend = (
        service.backend(HARDWARE_BACKEND)
        if HARDWARE_BACKEND
        else service.least_busy(
            operational=True, simulator=False, min_num_qubits=HARDWARE_MIN_QUBITS
        )
    )
    pass_manager = generate_preset_pass_manager(
        backend=backend, optimization_level=HARDWARE_OPTIMIZATION_LEVEL
    )
    print(f"Hardware mode: {backend.name}, {HARDWARE_SHOTS} shots per sampler call")
    if not LOAD_WARM_START:
        print("Warning: hardware mode is starting without simulator-optimized parameters.")
else:
    print("Simulator mode: Aer SamplerV2")

def prepare_sampler(logical_ansatz, simulator_shots, seed=None):
    """Return a measured sampling circuit and matching SamplerV2."""
    measured = logical_ansatz.copy()
    measured.measure_all()
    if EXECUTION_MODE == "simulator":
        return measured, AerSampler(default_shots=simulator_shots, seed=seed)
    isa_circuit = pass_manager.run(measured)
    sampler = RuntimeSampler(mode=backend)
    sampler.options.default_shots = HARDWARE_SHOTS
    return isa_circuit, sampler

def validated_warm_start(name, expected_metadata, parameter_count):
    """Return saved parameters after strict model/ansatz validation."""
    if not LOAD_WARM_START:
        return None
    for key, expected in expected_metadata.items():
        actual = warm_start_metadata.get(key)
        if actual != expected:
            raise ValueError(
                f"Warm start mismatch for {key}: expected {expected!r}, got {actual!r}"
            )
    values = warm_start.get(name)
    if values is None or values.shape != (parameter_count,):
        raise ValueError(
            f"Warm start {name!r} must contain {parameter_count} parameters"
        )
    return values.copy()

In [ ]:
# Autoload modules
%load_ext autoreload
%autoreload 2

In [ ]:
# Necessary imports
from fcc import MiyazawaJerniganInteraction, Peptide, ProteinFoldingProblem, PenaltyParameters, ProteinSolver, ProteinFoldingResult, ProteinShapeDecoder, build_turn_only_fcc_model
import fcc
from vqe import ChanceConstrainedVQEC
from qiskit.circuit.library import real_amplitudes
from qiskit_aer.primitives import SamplerV2 as Sampler
import matplotlib.pyplot as plt
import psutil
import numpy as np
from time import time

In [ ]:
# Resolve the cross-platform strategy selected in the configuration cell.
from fcc.measurement_utils import resolve_parallelizer

num_workers = NUM_WORKERS or psutil.cpu_count(logical=False) or 1
parallelizer = resolve_parallelizer(PARALLELIZER)
if parallelizer == "ray":
    import ray

    ray.init(
        num_cpus=num_workers,
        ignore_reinit_error=True,
        log_to_driver=False,
        runtime_env={"py_modules": [fcc]},
    )
print(f"Parallelizer: {parallelizer}; workers: {num_workers}")

## PolyFit

In [ ]:
protein_seq = POLYFIT_PROTEIN_SEQUENCE
penalty_back = POLYFIT_PENALTY_BACK
penalty_redun = POLYFIT_PENALTY_REDUNDANCY
penalty_olap = POLYFIT_PENALTY_OVERLAP

# Build the peptide object
peptide = Peptide(protein_seq)
# Set up the interaction and penalty terms
mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file="mj_matrix")
penalty_terms = PenaltyParameters(
    penalty_back=penalty_back, penalty_redun=penalty_redun, penalty_olap=penalty_olap
)
# Build the protein folding problem
pf_problem = ProteinFoldingProblem(
    peptide=peptide, interaction=mj_interaction, penalty_parameters=penalty_terms
)
# Get the Hamiltonian
qubit_op = pf_problem.qubit_op(
    r2_threshold=POLYFIT_R2_THRESHOLD, chunk=POLYFIT_CHUNK_SIZE
)
print(f"Number of qubits: {qubit_op.num_qubits}")
print(f"Number of terms in the Hamiltonian: {qubit_op.size}")

In [ ]:
# Choose the ansatz for the VQE
ansatz = real_amplitudes(
    qubit_op.num_qubits, reps=POLYFIT_ANSATZ_REPS, entanglement="linear"
)
sampling_ansatz, sampler = prepare_sampler(ansatz, POLYFIT_SHOTS)
polyfit_initial_params = validated_warm_start(
    "polyfit_params",
    {
        "polyfit_sequence": POLYFIT_PROTEIN_SEQUENCE,
        "polyfit_qubits": qubit_op.num_qubits,
        "polyfit_ansatz_reps": POLYFIT_ANSATZ_REPS,
    },
    ansatz.num_parameters,
)
# Set up the solver
optimizer = POLYFIT_OPTIMIZER
max_iter = (
    POLYFIT_SIMULATOR_MAX_ITER
    if EXECUTION_MODE == "simulator"
    else POLYFIT_HARDWARE_MAX_ITER
)
num_batches = num_workers  # Parallelize the energy evaluations
protein_solver = ProteinSolver(
    ansatz=sampling_ansatz,
    hamiltonian=qubit_op,
    sampler=sampler,
    parallelizer=parallelizer,
)
# Run the VQE
start_time = time()
polyfit_result = protein_solver.train(
    optimizer=optimizer,
    maxiter=max_iter,
    num_batches=num_batches,
    init_params=polyfit_initial_params,
)
end_time = time()
print(f"VQE completed in {end_time - start_time:.2f} seconds")
print(f"VQE result keys: {polyfit_result.keys()}")

In [ ]:
polyfit_result["top_solutions"][:5]

In [ ]:
# Plot the top 5 solutions
for i, (bitstring, energy) in enumerate(polyfit_result["top_solutions"][:5]):
    pf_result = ProteinFoldingResult(
        peptide=peptide,
        unused_qubits=pf_problem.unused_qubits,
        solution_bitstring=bitstring,
    )
    fig = pf_result.get_figure(
        title=f"Best solution {i+1}: {bitstring} (Energy: {energy:.2f})"
    )

## Turn-only chance-constrained VQEC

The revised VQEC path uses only the compact FCC turn register. Each sampled bitstring is decoded and scored with $H_{back} + H_{redun} + \sum_{m,n} \epsilon_{mn} \mathbf{1}[D_{mn}=2]$. One explicit chance constraint $\Pr[D_{mn}=0]-\delta_{mn} \leq 0$ is imposed for each residue pair separated by at least three sequence positions. No contact ancillas or optimizer-side postselection are used.

In [ ]:
protein_seq = VQEC_PROTEIN_SEQUENCE
penalty_back = VQEC_PENALTY_BACK
penalty_redun = VQEC_PENALTY_REDUNDANCY

# Build the peptide object
peptide = Peptide(protein_seq)
# Set up the interaction and penalty terms
mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file="mj_matrix")
penalty_terms = PenaltyParameters(
    penalty_back=penalty_back, penalty_redun=penalty_redun
)
# Build the compact sample-scored model. Geometry maps and contact ancillas
# from the historical symbolic path are not constructed.
turn_only_model = build_turn_only_fcc_model(
    peptide, interaction=mj_interaction, penalty_parameters=penalty_terms
)
delta_mn = np.full(turn_only_model.constraint_count, VQEC_CONSTRAINT_LIMIT)
print(f"Number of turn qubits: {turn_only_model.num_qubits}")
print("Chance-constraint limits:")
for pair, limit in zip(turn_only_model.constrained_pairs, delta_mn):
    print(f"  {pair}: {limit}")

In [ ]:
# Use an unmeasured compact-register ansatz. The solver adds measurement
# to its own copy and evaluates diagonal primitives from every sample.
ansatz = real_amplitudes(
    turn_only_model.num_qubits, reps=VQEC_ANSATZ_REPS, entanglement="linear"
)
shots = VQEC_SHOTS if EXECUTION_MODE == "simulator" else HARDWARE_SHOTS
sampling_ansatz, sampler = prepare_sampler(
    ansatz, VQEC_SHOTS, seed=VQEC_RANDOM_SEED
)

# Hyperparameters for the VQEC optimization
initial_params = validated_warm_start(
    "vqec_primal_params",
    {
        "vqec_sequence": VQEC_PROTEIN_SEQUENCE,
        "vqec_qubits": turn_only_model.num_qubits,
        "vqec_ansatz_reps": VQEC_ANSATZ_REPS,
        "vqec_constraint_count": turn_only_model.constraint_count,
    },
    ansatz.num_parameters,
)
if initial_params is None:
    initial_params = np.random.default_rng(VQEC_RANDOM_SEED).uniform(
        0, 2 * np.pi, ansatz.num_parameters
    )
init_dual_vars = warm_start.get("vqec_dual_vars") if LOAD_WARM_START else None
if init_dual_vars is None:
    init_dual_vars = np.zeros(turn_only_model.constraint_count)
elif init_dual_vars.shape != (turn_only_model.constraint_count,):
    raise ValueError("Warm-start VQEC dual-variable count is incompatible")
perturb_step = VQEC_PERTURB_STEP
gamma = VQEC_GAMMA
max_iter = (
    VQEC_SIMULATOR_MAX_ITER
    if EXECUTION_MODE == "simulator"
    else VQEC_HARDWARE_MAX_ITER
)

vqec_solver = ChanceConstrainedVQEC(
    turn_only_model,
    ansatz,
    sampler,
    constraint_limits=delta_mn,
    shots=shots,
    sampling_circuit=sampling_ansatz,
)
vqec_result = vqec_solver.optimize_primal_dual(
    initial_params=initial_params,
    initial_dual_vars=init_dual_vars,
    primal_perturb_step=perturb_step,
    dual_perturb_step=perturb_step,
    gamma=gamma,
    auto_update_step=False,
    max_iter=max_iter,
)

In [ ]:
vqec_result.keys()

In [ ]:
# Plot the energy and constraint expectations
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(vqec_result["energy_history"], label="Energy", marker=".")
ax[0].set_xlabel("Iteration")
ax[0].set_ylabel("Energy expectation")

for i, pair in enumerate(turn_only_model.constrained_pairs):
    ax[1].plot(
        np.array(vqec_result["constraints_history"])[:, i],
        label=f"Pair {pair}",
        marker=".",
    )
ax[1].set_xlabel("Iteration")
ax[1].set_ylabel(r"Chance residual $\Pr[D_{mn}=0]-\delta_{mn}$")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sample the optimized circuit through the same unfiltered solver path.
counts = vqec_solver.sample_counts(vqec_result["optimal_primal_vars"])[0]

# Sort the binary strings by their counts
sorted_quasi_dist = dict(sorted(counts.items(), key=lambda item: item[1], reverse=True))
# Score the most frequent samples directly from their decoded geometries.
bitstring_energies = {}
for i, (sample, count) in enumerate(sorted_quasi_dist.items()):
    if i >= 5:
        break
    evaluation = turn_only_model.evaluate_bitstring(sample)
    bitstring_energies[sample] = evaluation.objective
    print(
        f"Sample {i}: {sample}, probability={count / shots:.4f}, "
        f"objective={evaluation.objective:.4f}, "
        f"overlaps={evaluation.overlap_indicators.astype(int).tolist()}"
    )

In [ ]:
# Invalid turn codes remain part of optimization statistics; only the
# optional structure visualization below requires a physical encoding.
for i, (sample, energy) in enumerate(bitstring_energies.items()):
    if not turn_only_model.evaluate_bitstring(sample).physical_encoding:
        print(f"Skipping visualization of nonphysical turn code: {sample}")
        continue
    pf_result = ProteinFoldingResult(
        peptide=peptide,
        unused_qubits=[],
        solution_bitstring=sample,
    )
    fig = pf_result.get_figure(
        title=f"Most frequent solution {i+1}: {sample} (Energy: {energy:.2f})"
    )
    plt.show()

## Comparison to classical exhaustive search results

In [ ]:
from fcc.classical_utils import load_top_cls_solns

file_name = "topobj_GNLVS.txt"
top_cls_solutions = load_top_cls_solns(file_name)
top_cls_solutions[:5]

In [ ]:
# Example: Translate the top polyfit solutions to turn sequences
top_polyfit_solutions = polyfit_result["top_solutions"][:5]
print(top_polyfit_solutions)

polyfit_turns = []
for i, (bitstring, energy) in enumerate(top_polyfit_solutions):
    pf_decoder = ProteinShapeDecoder(
        peptide=peptide,
        solution_bitstring=bitstring,
    )
    polyfit_turns.append([pf_decoder.turn_sequence, energy])

polyfit_turns

In [ ]:
# Save a validated simulator result for a later hardware refinement run.
if SAVE_WARM_START:
    from importlib.metadata import version

    metadata = {
        "format_version": 1,
        "polyfit_sequence": POLYFIT_PROTEIN_SEQUENCE,
        "polyfit_qubits": qubit_op.num_qubits,
        "polyfit_ansatz_reps": POLYFIT_ANSATZ_REPS,
        "vqec_sequence": VQEC_PROTEIN_SEQUENCE,
        "vqec_qubits": turn_only_model.num_qubits,
        "vqec_ansatz_reps": VQEC_ANSATZ_REPS,
        "vqec_constraint_count": turn_only_model.constraint_count,
        "qiskit_version": version("qiskit"),
        "qiskit_aer_version": version("qiskit-aer"),
        "qiskit_algorithms_version": version("qiskit-algorithms"),
    }
    np.savez_compressed(
        warm_start_path,
        metadata=json.dumps(metadata, sort_keys=True),
        polyfit_params=np.asarray(polyfit_result["opt_params"], dtype=float),
        vqec_primal_params=np.asarray(
            vqec_result["optimal_primal_vars"], dtype=float
        ).reshape(-1),
        vqec_dual_vars=np.asarray(
            vqec_result["optimal_dual_vars"], dtype=float
        ).reshape(-1),
    )
    print(f"Saved warm start to {warm_start_path}")